# Explicit sklearn Logistic Plugin

This notebook keeps estimator code outside core Aegis and registers it explicitly before config validation. YAML only references `model.plugin_id`.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
from typing import Any, Mapping

import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from research.aegis_research.config import ModelConfig, load_experiment_config, resolve_experiment_config
from research.aegis_research.experiments import run_experiment
from research.aegis_research.model_contracts import (
    POSITIVE_CLASS_PROBABILITY,
    ModelDataset,
    ModelExecutionContext,
    ModelFitResult,
    ModelPluginDeclaration,
    ModelPluginDefinition,
    ModelPredictionResult,
)
from research.aegis_research.model_export import export_model_bundle
from research.aegis_research.model_registry import ModelRegistry


In [ ]:
class SklearnLogisticPlugin:
    def fit(self, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelFitResult:
        del context
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                max_iter=int(params.get('max_iter', 1000)),
                random_state=int(params.get('random_state', 42)),
            )),
        ])
        if dataset.target is None:
            raise ValueError('fit dataset target is required')
        model.fit(dataset.features, dataset.target.astype(int))
        classes = tuple(model.named_steps['classifier'].classes_)
        return ModelFitResult(
            state={'model': model},
            observed_classes=classes,
            class_probability_columns={class_label: f'class_{class_label}_probability' for class_label in classes},
            diagnostics={'estimator': 'sklearn.linear_model.LogisticRegression'},
            state_metadata={'state_format': 'pickle'},
        )

    def predict(self, state: Any, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelPredictionResult:
        del params, context
        model = state['model']
        classes = tuple(model.named_steps['classifier'].classes_)
        columns = {class_label: f'class_{class_label}_probability' for class_label in classes}
        return ModelPredictionResult(
            probabilities=pd.DataFrame(
                model.predict_proba(dataset.features),
                index=dataset.row_index,
                columns=[columns[class_label] for class_label in classes],
            ),
            observed_classes=classes,
            class_probability_columns=columns,
        )


In [ ]:
def validate_params(params: Mapping[str, Any]) -> Mapping[str, str]:
    issues = {}
    if 'max_iter' in params and (not isinstance(params['max_iter'], int) or params['max_iter'] <= 0):
        issues['max_iter'] = 'must be a positive integer'
    if 'random_state' in params and not isinstance(params['random_state'], int):
        issues['random_state'] = 'must be an integer'
    return issues

def build_registry():
    registry = ModelRegistry()
    registry.register(ModelPluginDefinition(
        declaration=ModelPluginDeclaration(
            id='examples.sklearn_logistic',
            version='1.0.0',
            prediction_outputs=(POSITIVE_CLASS_PROBABILITY,),
            state_schema_version='example_sklearn_logistic_state.v1',
            package_versions={'sklearn': sklearn.__version__},
        ),
        plugin=SklearnLogisticPlugin(),
        validate_params=validate_params,
    ))
    return registry.freeze()


In [ ]:
registry = build_registry()
loaded = load_experiment_config('research/configs/experiments/synthetic_purged_fixlb_baseline.yaml')
experiment = replace(
    loaded.config,
    model=ModelConfig(
        plugin_id='examples.sklearn_logistic',
        min_train_samples=50,
        params={'max_iter': 1000, 'random_state': 42},
    ),
    output_dir='runs/example_model_plugin',
)
resolved = resolve_experiment_config(experiment, model_registry=registry)
result = run_experiment(resolved, run_id='sklearn-logistic-plugin-example')
result['status']


In [ ]:
# Optional producer-side export for another prediction-only runtime project.
# The consuming project must register reviewed plugin code and validate this metadata before loading native state.
export_model_bundle(
    result['run_dir'],
    model_artifact_id='validation.split_0.model',
    output_dir=Path(result['run_dir']) / 'exports' / 'split_0_model',
)
